In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import r2_score
import random
from scipy.signal import savgol_filter
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter

In [ ]:
# Function to estimate log-normal distribution parameters from mean and median
def estimate_lognormal_params(mean, median):
    sigma = np.sqrt(2 * np.log(mean / median))
    mu = np.log(median)
    return mu, sigma

# Function to generate synthetic data using log-normal distribution and clip to observed range
def generate_lognormal_samples(mean, median, min_val, max_val, size=5000):
    mu, sigma = estimate_lognormal_params(mean, median)
    samples = np.random.lognormal(mean=mu, sigma=sigma, size=size)
    return np.clip(samples, min_val, max_val)


def generate_time (t_initial, t_final, n_timesteps):
    t_numpy = np.linspace(t_initial, t_final, n_timesteps)
    t_reshape = t_numpy.reshape(1, -1)
    t_torch = torch.tensor(t_reshape, dtype=torch.float32).to(device)
    return t_numpy, t_reshape, t_torch

def smooth_peak(curve, window_length=11, polyorder=3, baseline_ratio=0.01):
    curve = np.asarray(curve)
    smoothed = savgol_filter(curve, window_length=window_length, polyorder=polyorder)

    peak_idx = np.argmax(smoothed)
    peak_val = smoothed[peak_idx]
    threshold = peak_val * baseline_ratio

    # Find start (left of peak)
    start_idx = 0
    for i in range(peak_idx, 0, -1):
        if smoothed[i] < threshold:
            start_idx = i
            break

    # Find end (right of peak)
    end_idx = len(curve) - 1
    for i in range(peak_idx, len(curve)):
        if smoothed[i] < threshold:
            end_idx = i
            break

    # Keep only the main peak region
    cleaned = np.zeros_like(curve)
    cleaned[start_idx:end_idx + 1] = curve[start_idx:end_idx + 1]

    return cleaned

def zscore(x, eps=1e-12):
    x = np.asarray(x, dtype=float)
    mu = x.mean()
    sd = x.std()
    if sd < eps:
        return x * 0.0
    return (x - mu) / sd

def best_lag_correlation(a, b, max_lag=None, normalize='zscore'):

    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    T = min(len(a), len(b))
    a = a[:T]
    b = b[:T]

    if normalize == 'zscore':
        a_n = zscore(a)
        b_n = zscore(b)
    else:
        a_n = a
        b_n = b

    if max_lag is None:
        max_lag = T - 1

    best_corr = -np.inf
    best_lag = 0
    best_pair = (None, None)

    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:

            a_slice = a_n[-lag:]
            b_slice = b_n[:T+lag]
        elif lag > 0:

            a_slice = a_n[:T-lag]
            b_slice = b_n[lag:]
        else:
            a_slice = a_n
            b_slice = b_n

        if len(a_slice) < 2:
            continue
        corr = np.dot(a_slice, b_slice) / (len(a_slice) - 1)

        if corr > best_corr:
            best_corr = corr
            best_lag = lag
            if lag < 0:
                a_raw = a[-lag:]
                b_raw = b[:T+lag]
            elif lag > 0:
                a_raw = a[:T-lag]
                b_raw = b[lag:]
            else:
                a_raw = a
                b_raw = b
            best_pair = (a_raw, b_raw)

    return {'lag': best_lag, 'corr': best_corr, 'a_seg': best_pair[0], 'b_seg': best_pair[1]}

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Generate synthetic values for each variable
mass     = generate_lognormal_samples(mean=9.62,     median=2.81,     min_val=0.0404,    max_val=108.86)
area     = generate_lognormal_samples(mean=224.82,   median=17.09,    min_val=0.98,      max_val=4931.81)
velocity = generate_lognormal_samples(mean=0.51,     median=0.40,     min_val=0.04,      max_val=1.58)
distance = generate_lognormal_samples(mean=46162.12, median=29611.86, min_val=35.41,     max_val=294509.22)
dispersion = generate_lognormal_samples(mean=130.73,     median=37.48,     min_val=1.9,      max_val=1486.45)

In [ ]:
# Linear scale and reshape
M = mass.reshape(-1,1)
A = area.reshape(-1,1)
x = distance.reshape(-1,1)
u = velocity.reshape(-1,1)
Q = np.array(A*u)
Q = Q.reshape(-1,1)
R= 1
theta = 1
D = dispersion.reshape(-1,1)

In [ ]:
n_timesteps = 100000
t_numpy, t_reshape, t_torch = generate_time(1, 3e6, n_timesteps)
c_downstream_filter = (1e6 * M) / (2 * theta * A * R * np.sqrt(np.pi * D * t_reshape/ R)) * np.exp(-((x - u * t_reshape / R) ** 2) / (4 * D * t_reshape / R))

In [ ]:
t_peak = []
max_concentrion_array = []
for i in range (c_downstream_filter.shape[0]):
  max_concentrion_idx = np.argmax(c_downstream_filter[i,:])
  max_concentrion = np.max(c_downstream_filter[i,:])
  time_peak = t_reshape[:,max_concentrion_idx]
  t_peak = np.append(t_peak,time_peak)
  max_concentrion_array = np.append(max_concentrion_array,max_concentrion)

v = distance/t_peak
pe = (distance*velocity)/dispersion
valid_v = (v >= 0.1) & (v <= 6)
valid_pe = (pe >= 10) & (pe <= 1500)
valid_max_c = (max_concentrion_array >= 0.3) & (max_concentrion_array <= 3000)
valid_ratio =  valid_v & valid_pe & valid_max_c

In [ ]:
# Linear scale and reshape
valid_M = M[valid_ratio]
valid_M = valid_M[0:3000]
valid_A = A[valid_ratio]
valid_A = valid_A[0:3000]
valid_x = x[valid_ratio]
valid_x = valid_x[0:3000]
x_upstream = np.array(valid_x/ 10)
x_upstream = x_upstream.reshape(-1,1)
valid_u = u[valid_ratio]
valid_u = valid_u[0:3000]
valid_D = D[valid_ratio]
valid_D = valid_D[0:3000]
valid_Q = np.array(valid_A*valid_u)

In [ ]:
# Calculate upstream and downstream BTCs using ADE analytical solution
c_upstream = (1e6 * valid_M) / (2 * theta * valid_A * R * np.sqrt(np.pi * valid_D * t_reshape / R)) * np.exp(-((x_upstream - valid_u * t_reshape / R) ** 2) / (4 * valid_D * t_reshape / R))

c_downstream = (1e6 * valid_M) / (2 * theta * valid_A * R * np.sqrt(np.pi * valid_D * t_reshape/ R)) * np.exp(-((valid_x - valid_u * t_reshape / R) ** 2) / (4 * valid_D * t_reshape / R))

In [ ]:
t_max = []
t_peak = []
c_downstream_train = c_downstream[0:2500,:]

c_upstream_train = c_upstream[0:2500,:] # input
c_downstream_truncated = c_downstream_train.copy() # target

c_upstream_test = c_upstream[2501:,:] # input for test non truncated
c_downstream_test = c_downstream[2501:,:] # target for test non truncated

for i in range(c_downstream_train.shape[0]):
    # Find peak index and time
    c_peak_idx = np.argmax(c_downstream[i])
    t_peak_val = t_numpy[c_peak_idx]
    t_peak.append(t_peak_val)

    t_max_idx = int(1.2 * c_peak_idx)
    t_max.append(t_numpy[t_max_idx] if t_max_idx < len(t_numpy) else t_numpy[-1])
    truncated = c_downstream[i, :t_max_idx]
    c_last = 0

    pad_length = c_downstream.shape[1] - t_max_idx
    padding = np.full(pad_length, c_last, dtype=truncated.dtype)

    full_curve = np.concatenate([truncated, padding])
    c_downstream_truncated[i] = full_curve

In [ ]:
dtype = torch.float32
X = torch.tensor(np.array(c_upstream_train)).to(device, dtype)
y = torch.tensor(np.array(c_downstream_truncated)).to(device,dtype)

x_test_nontruncated = torch.tensor(np.array(c_upstream_test)).to(device, dtype)
y_test_nontruncated = torch.tensor(np.array(c_downstream_test)).to(device,dtype)

In [ ]:
dataset = TensorDataset(X, y)
total_size = len(dataset)
train_size = int(0.7 * total_size)
val_size = int(0.2 * total_size)
test_size = total_size - train_size - val_size


train_dataset, test_dataset, val_dataset = random_split(dataset, [train_size, test_size, val_size])
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Define the MLP model
class MLP(nn.Module):
    def __init__(self, input_size, output_size):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 2048),
            nn.ReLU(),

            nn.Linear(2048, 1024),
            nn.ReLU(),

            nn.Linear(1024, output_size)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
# Model, loss, optimizer
input_size = X.shape[1]
output_size = y.shape[1]
model = MLP(input_size, output_size).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001,weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=200, gamma=0.5)

In [ ]:
train_losses = []
val_losses = []

epochs = 800  # Set number of epochs

for epoch in range(epochs):
    model.train()  # Set model to training mode
    running_loss = 0.0  # Track training loss

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()

    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)  # Store training loss

    # Print training loss every 100 epochs
    if epoch % 100 == 0:
        print(f'Epoch {epoch}, Training Loss: {avg_train_loss:.6f}')

    # Validation step every 30 epochs
    if epoch % 30 == 0:
        model.eval()  # Set model to evaluation mode
        val_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        val_losses.append((epoch, avg_val_loss))

        print(f'Epoch {epoch}, Validation Loss: {avg_val_loss:.6f}')

In [ ]:
val_epochs, val_loss_values = zip(*val_losses)

# Plot Training and Validation Loss
plt.figure(figsize=(10, 6))
plt.semilogy(range(epochs), train_losses, label='Training Loss', color='blue')
plt.semilogy(val_epochs, val_loss_values, label='Validation Loss', color='red', marker='o')

plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Test with truncated data
model.eval()

actual_curves = []
predicted_curves = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = model(batch_X)

        actual_curves.extend(batch_y.cpu().numpy())
        predicted_curves.extend(outputs.cpu().numpy())

actual_curves = np.array(actual_curves)
predicted_curves = np.array(predicted_curves)
predicted_curves[predicted_curves < 0] = 0

In [ ]:
max_lag = 100
use_normalize = 'zscore'

cc_results = []
for i in range(actual_curves.shape[0]):
    a = actual_curves[i]
    b = smooth_peak(predicted_curves[i])
    res = best_lag_correlation(a, b, max_lag=max_lag, normalize=use_normalize)
    cc_results.append(res)

cc_mean_corr = np.mean([r['corr'] for r in cc_results])
print(f"[Cross-Correlation]: {cc_mean_corr:.4f}")

In [ ]:
# Peak concentration
actual_peak = []
predicted_peak = []

for i in range(actual_curves.shape[0]):
    actual_peak.append(np.max(actual_curves[i, :]))
    predicted_peak.append(np.max(predicted_curves[i, :]))

actual_peak = np.array(actual_peak)
predicted_peak = np.array(predicted_peak)

# R2
r2_peak = r2_score(actual_peak, predicted_peak)
print(f'R² Score for Peak Concentration: {r2_peak:.6f}')

# MAPE
mape = np.mean(np.abs((predicted_peak - actual_peak) / actual_peak)) * 100
print("MAPE (%):", mape)

# BAIS RATIO
y_true = actual_peak
y_pred = predicted_peak
bias_ratio = np.mean(y_pred) / np.mean(y_true)
print("Bias Ratio:", bias_ratio)
x1 = np.linspace(0, np.max(actual_peak))
y1=x1
plt.figure(figsize=(10, 6))
plt.scatter(actual_peak,predicted_peak)
plt.plot(x1,y1, linestyle='--', color='r')
plt.yscale('log')
plt.xscale('log')
plt.ylabel('Predicted Peak Concentration (ppb)')
plt.xlabel('Actual Peak Concentration (ppb)')
plt.grid(True)
plt.show()

In [ ]:
# Peak time
actual_peaktime = []
predicted_peaktime = []

for i in range(actual_curves.shape[0]):
    actual_peaktime.append(t_numpy[np.argmax(actual_curves[i, :])])
    predicted_peaktime.append(t_numpy[np.argmax(predicted_curves[i, :])])

# Convert to arrays
actual_peaktime = np.array(actual_peaktime)
predicted_peaktime = np.array(predicted_peaktime)

# R2
r2_peaktime = r2_score(actual_peaktime, predicted_peaktime)
print(f'R² Score for Peak Time: {r2_peaktime:.6f}')

#MAPE
mape = np.mean(np.abs((predicted_peaktime - actual_peaktime) / actual_peaktime)) * 100
print("MAPE (%):", mape)

# BAIAS RATIO
y_true = actual_peaktime
y_pred = predicted_peaktime
bias_ratio = np.mean(y_pred) / np.mean(y_true)
print(f"Bias ratio: {bias_ratio:.3f}")

x2 = np.linspace(0, np.max(actual_peaktime)+100)
y2=x2
plt.figure(figsize=(10, 6))
plt.scatter(actual_peaktime,predicted_peaktime)
plt.plot(x2,y2,linestyle='--', color='r')
plt.ylabel('Predicted Peak Time (s)')
plt.xlabel('Actual Peak Time (s)')
plt.yscale('log')
plt.xscale('log')
plt.grid(True)
plt.show()

In [ ]:
actual_m0 =[]
predicted_m0 =[]

for i in range(actual_curves.shape[0]):
    actual_m0.append(np.trapezoid(actual_curves[i,:],t_reshape))
    predicted_m0.append(np.trapezoid(smooth_peak(predicted_curves[i,:]),t_reshape))

actual_m0 = np.array(actual_m0)
predicted_m0 = np.array(predicted_m0)

# R2
r2_m0 = r2_score(actual_m0, predicted_m0)
print(f'R² Score for M0: {r2_m0:.6f}')


# MAPE
mape = np.mean(np.abs((predicted_m0 - actual_m0) / actual_m0)) * 100
print("MAPE (%):", mape)

# BAIAS RATIO
y_true = actual_m0
y_pred = predicted_m0
bias_ratio = np.mean(y_pred) / np.mean(y_true)
print(f"Bias ratio: {bias_ratio:.3f}")

x3 = np.linspace(0, np.max(actual_m0)+100)
y3=x3
plt.figure(figsize=(10, 6))
plt.scatter(actual_m0,predicted_m0)
plt.plot(x3,y3,linestyle='--', color='r')
plt.ylabel('Predicted M0')
plt.xlabel('Actual M0')
plt.yscale('log')
plt.xscale('log')
plt.grid(True)
plt.show()

In [ ]:
actual_m1 = []
predicted_m1 = []

for i in range(actual_curves.shape[0]):
    actual_moment = actual_curves[i,:]*t_reshape
    predicted_moment = smooth_peak(predicted_curves[i,:])*t_reshape

    actual_m1.append(np.trapezoid(actual_moment,t_reshape))
    predicted_m1.append(np.trapezoid(predicted_moment,t_reshape))

actual_m1 = np.array(actual_m1)
predicted_m1 = np.array(predicted_m1)

# R2
r2_m1 = r2_score(actual_m1,predicted_m1)
print(f'R² Score for M1: {r2_m1:.6f}')

# MAPE
mape = np.mean(np.abs((predicted_m1 - actual_m1) / actual_m1)) * 100
print("MAPE (%):", mape)

# BAIAS RATIO
y_true = actual_m1
y_pred = predicted_m1
bias_ratio = np.mean(y_pred) / np.mean(y_true)
print(f"Bias ratio: {bias_ratio:.3f}")
x4 = np.linspace(0, np.max(actual_m1)+100)
y4=x4
plt.figure(figsize=(10, 6))
plt.scatter(actual_m1,predicted_m1)
plt.plot(x4,y4,linestyle='--', color='r')
plt.ylabel('Predicted M1')
plt.xlabel('Actual M1')
plt.grid(True)
plt.yscale('log')
plt.xscale('log')
plt.show()

# TEST NON-TRUNCATED DATA

In [ ]:
# TEST NON-TRUNCATED DATA

model.eval()
actual_curves_nontruncated = []
predicted_curves_nontruncated  = []

with torch.no_grad():
      outputs_test = model(x_test_nontruncated)

      actual_curves_nontruncated.extend(y_test_nontruncated.cpu().numpy())
      predicted_curves_nontruncated.extend(outputs_test.cpu().numpy())

actual_curves_nontruncated = np.array(actual_curves_nontruncated)
predicted_curves_nontruncated = np.array(predicted_curves_nontruncated)
predicted_curves_nontruncated[predicted_curves_nontruncated < 0] = 0

In [ ]:
max_lag = 100
use_normalize = 'zscore'

cc_results = []
for i in range(actual_curves_nontruncated.shape[0]):
    a = actual_curves_nontruncated[i]
    b = smooth_peak(predicted_curves_nontruncated[i])
    res = best_lag_correlation(a, b, max_lag=max_lag, normalize=use_normalize)
    cc_results.append(res)

cc_mean_corr = np.mean([r['corr'] for r in cc_results])
print(f"[Cross-Correlation]: {cc_mean_corr:.4f}")

In [ ]:
num_curves_to_plot = len(actual_curves_nontruncated)
plt.figure(figsize=(10, 6))

for i in range(50):
    plt.semilogx(actual_curves_nontruncated[i], label=f'True {i+1}', linestyle='dashed')
    plt.semilogx(smooth_peak(predicted_curves_nontruncated[i]), label=f'Predicted {i+1}', linestyle='solid')
    plt.xlabel('Time Index')
    plt.ylabel('Concentration')
    plt.title('Tested Breakthrough Curves: True vs Predicted TEST')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Peak concentration
actual_peak = []
predicted_peak = []

for i in range(actual_curves_nontruncated.shape[0]):
    actual_peak.append(np.max(actual_curves_nontruncated[i, :]))
    predicted_peak.append(np.max(predicted_curves_nontruncated[i, :]))

actual_peak = np.array(actual_peak)
predicted_peak = np.array(predicted_peak)

# R2
r2_peak = r2_score(actual_peak, predicted_peak)
print(f'R² Score for Peak Concentration: {r2_peak:.6f}')


# MAPE
mape = np.mean(np.abs((predicted_peak - actual_peak) / actual_peak)) * 100
print("MAPE (%):", mape)

# BAIS RATIO
y_true = actual_peak
y_pred = predicted_peak
bias_ratio = np.mean(y_pred) / np.mean(y_true)
print("Bias Ratio:", bias_ratio)
x1 = np.linspace(0, np.max(actual_peak))
y1=x1
plt.figure(figsize=(10, 6))
plt.scatter(actual_peak,predicted_peak)
plt.plot(x1,y1, linestyle='--', color='r')
plt.ylabel('Predicted Peak Concentration (ppb)')
plt.xlabel('Actual Peak Concentration (ppb)')
plt.grid(True)

plt.yscale('log')
plt.xscale('log')
plt.show()

In [ ]:
# Peak time
actual_peaktime = []
predicted_peaktime = []

for i in range(actual_curves_nontruncated.shape[0]):
    actual_peaktime.append(t_numpy[np.argmax(actual_curves_nontruncated[i, :])])
    predicted_peaktime.append(t_numpy[np.argmax(predicted_curves_nontruncated[i, :])])

# Convert to arrays
actual_peaktime = np.array(actual_peaktime)
predicted_peaktime = np.array(predicted_peaktime)

# R2
r2_peaktime = r2_score(actual_peaktime, predicted_peaktime)
print(f'R² Score for Peak Time: {r2_peaktime:.6f}')

#MAPE
mape = np.mean(np.abs((predicted_peaktime - actual_peaktime) / actual_peaktime)) * 100
print("MAPE (%):", mape)

# BAIAS RATIO
y_true = actual_peaktime
y_pred = predicted_peaktime
bias_ratio = np.mean(y_pred) / np.mean(y_true)
print(f"Bias ratio: {bias_ratio:.3f}")

x2 = np.linspace(0, np.max(actual_peaktime)+100)
y2=x2
plt.figure(figsize=(10, 6))
plt.scatter(actual_peaktime,predicted_peaktime)
plt.plot(x2,y2,linestyle='--', color='r')
plt.ylabel('Predicted Peak Time (s)')
plt.xlabel('Actual Peak Time (s)')
plt.grid(True)

plt.yscale('log')
plt.xscale('log')
plt.show()

In [ ]:
actual_m0 =[]
predicted_m0 =[]

for i in range(actual_curves_nontruncated.shape[0]):
    actual_m0.append(np.trapezoid(actual_curves_nontruncated[i,:],t_reshape))
    predicted_m0.append(np.trapezoid(smooth_peak(predicted_curves_nontruncated[i,:]),t_reshape))

actual_m0 = np.array(actual_m0)
predicted_m0 = np.array(predicted_m0)

# R2
r2_m0 = r2_score(actual_m0, predicted_m0)
print(f'R² Score for M0: {r2_m0:.6f}')

# MAPE
mape = np.mean(np.abs((predicted_m0 - actual_m0) / actual_m0)) * 100
print("MAPE (%):", mape)

# BAIAS RATIO
y_true = actual_m0
y_pred = predicted_m0
bias_ratio = np.mean(y_pred) / np.mean(y_true)
print(f"Bias ratio: {bias_ratio:.3f}")

x3 = np.linspace(0, np.max(actual_m0)+100)
y3=x3
plt.figure(figsize=(10, 6))
plt.scatter(actual_m0,predicted_m0)
plt.plot(x3,y3,linestyle='--', color='r')
plt.ylabel('Predicted M0')
plt.xlabel('Actual M0')
plt.grid(True)
plt.yscale('log')
plt.xscale('log')
plt.show()

In [ ]:
actual_m1 = []
predicted_m1 = []

for i in range(actual_curves_nontruncated.shape[0]):
    actual_moment = actual_curves_nontruncated[i,:]*t_reshape
    predicted_moment = smooth_peak(predicted_curves_nontruncated[i,:])*t_reshape

    actual_m1.append(np.trapezoid(actual_moment,t_reshape))
    predicted_m1.append(np.trapezoid(predicted_moment,t_reshape))

actual_m1 = np.array(actual_m1)
predicted_m1 = np.array(predicted_m1)

# R2
r2_m1 = r2_score(actual_m1,predicted_m1)
print(f'R² Score for M1: {r2_m1:.6f}')

# MAPE
mape = np.mean(np.abs((predicted_m1 - actual_m1) / actual_m1)) * 100
print("MAPE (%):", mape)

# BAIAS RATIO
y_true = actual_m1
y_pred = predicted_m1
bias_ratio = np.mean(y_pred) / np.mean(y_true)
print(f"Bias ratio: {bias_ratio:.3f}")

x4 = np.linspace(0, np.max(actual_m1)+100)
y4=x4
plt.figure(figsize=(10, 6))
plt.scatter(actual_m1,predicted_m1)
plt.plot(x4,y4,linestyle='--', color='r')
plt.ylabel('Predicted M1')
plt.xlabel('Actual M1')
plt.grid(True)
plt.yscale('log')
plt.xscale('log')
plt.show()